# 📈 Task 4: Sales Prediction using Python
### Predict Sales from Advertising Spend (TV, Radio, Newspaper)
**Dataset:** [Advertising CSV – Kaggle](https://www.kaggle.com/datasets/bumba5341/advertisingcsv)

---

## Step 1: Install & Import Libraries

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn tensorflow statsmodels --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print('✅ Libraries imported!')
print(f'TensorFlow: {tf.__version__}')

## Step 2: Load Dataset

In [ ]:
# ── Option A: Upload from Kaggle ──
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv('advertising.csv')

# ── Option B: Direct URL load ──
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/advertising.csv'
try:
    df = pd.read_csv(url)
    print('✅ Loaded from URL')
except:
    # Synthetic fallback
    np.random.seed(42)
    n = 200
    TV        = np.random.uniform(0.7, 296, n)
    Radio     = np.random.uniform(0, 49, n)
    Newspaper = np.random.uniform(0.3, 114, n)
    Sales     = 2.9 + 0.045*TV + 0.19*Radio + 0.001*Newspaper + np.random.normal(0, 1.5, n)
    df = pd.DataFrame({'TV': TV, 'Radio': Radio, 'Newspaper': Newspaper, 'Sales': Sales})
    print('✅ Using synthetic fallback dataset')

# Drop unnamed index column if present
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

print('Shape:', df.shape)
df.head(10)

## Step 3: EDA – Understanding the Advertising Data

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Statistical Summary ===')
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Advertising Dataset – EDA', fontsize=16, fontweight='bold')

channels = ['TV', 'Radio', 'Newspaper']
colors   = ['#3498db', '#e74c3c', '#2ecc71']

# Scatter: each channel vs Sales
for i, (ch, col) in enumerate(zip(channels, colors)):
    axes[0, i].scatter(df[ch], df['Sales'], alpha=0.6, color=col)
    # Trend line
    z = np.polyfit(df[ch], df['Sales'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[ch].min(), df[ch].max(), 100)
    axes[0, i].plot(x_line, p(x_line), 'k--', linewidth=1.5)
    axes[0, i].set_title(f'{ch} Spend vs Sales')
    axes[0, i].set_xlabel(f'{ch} ($000s)'); axes[0, i].set_ylabel('Sales (units)')

# Distribution of each channel
for i, (ch, col) in enumerate(zip(channels, colors)):
    axes[1, i].hist(df[ch], bins=25, color=col, edgecolor='black', alpha=0.8)
    axes[1, i].set_title(f'{ch} Spend Distribution')
    axes[1, i].set_xlabel(f'{ch} ($000s)')

plt.tight_layout()
plt.show()

# Correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(), annot=True, fmt='.3f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## Step 4: Data Preprocessing & Feature Engineering

In [ ]:
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

# Feature Engineering: Interaction terms
df_eng = X.copy()
df_eng['TV_Radio']     = df_eng['TV'] * df_eng['Radio']
df_eng['TV_sq']        = df_eng['TV'] ** 2
df_eng['Radio_sq']     = df_eng['Radio'] ** 2
df_eng['Total_Spend']  = df_eng['TV'] + df_eng['Radio'] + df_eng['Newspaper']
df_eng['TV_share']     = df_eng['TV'] / (df_eng['Total_Spend'] + 0.01)

X_eng = df_eng

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_eng, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print('Features:', list(X_eng.columns))

## Step 5: Statistical Analysis with Statsmodels

In [ ]:
X_sm = sm.add_constant(df[['TV', 'Radio', 'Newspaper']])
ols  = sm.OLS(y, X_sm).fit()
print(ols.summary())

print('\n=== Interpretation ===')
print('Coefficients show marginal contribution of each channel to sales.')
print('p-value < 0.05 = statistically significant predictor.')

## Step 6: Train & Compare ML Regression Models

In [ ]:
def evaluate(name, y_true, y_pred):
    return {'Model': name,
            'R2':   round(r2_score(y_true, y_pred), 4),
            'MAE':  round(mean_absolute_error(y_true, y_pred), 4),
            'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 4)}

models = {
    'Linear Regression':  LinearRegression(),
    'Ridge':              Ridge(alpha=1.0),
    'Lasso':              Lasso(alpha=0.01),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results  = []
trained  = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    res = evaluate(name, y_test, y_pred)
    results.append(res)
    trained[name] = y_pred
    print(f'{name:<25} | R²={res["R2"]:.4f} | MAE={res["MAE"]:.4f} | RMSE={res["RMSE"]:.4f}')

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
print('\nBest Model:', results_df.iloc[0]['Model'])

## Step 7: Advertising Impact Analysis

In [ ]:
best = RandomForestRegressor(n_estimators=100, random_state=42)
best.fit(X_train_s, y_train)
best_pred = best.predict(X_test_s)

# Feature importance
importances = pd.Series(best.feature_importances_, index=X_eng.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Sales Prediction – Model Results', fontsize=15, fontweight='bold')

# Feature importance
importances.plot(kind='bar', ax=axes[0], color='#3498db', edgecolor='black')
axes[0].set_title('Feature Importance (Random Forest)')
axes[0].set_ylabel('Importance Score')
axes[0].tick_params(axis='x', rotation=45)

# Actual vs Predicted
axes[1].scatter(y_test, best_pred, alpha=0.6, color='#e74c3c')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
axes[1].set_title('Actual vs Predicted Sales')
axes[1].set_xlabel('Actual Sales'); axes[1].set_ylabel('Predicted Sales')

# Model R² comparison
axes[2].barh(results_df['Model'], results_df['R2'], color='#2ecc71', edgecolor='black')
axes[2].set_xlabel('R²')
axes[2].set_title('Model R² Comparison')

plt.tight_layout()
plt.show()

print('\n=== Business Insights ===')
top3 = importances.head(3)
for feat, imp in top3.items():
    print(f'  • {feat:<20}: {imp*100:.1f}% importance')
print('\n→ Allocate more budget to top-performing channels for maximum ROI.')

## Step 8: Deep Learning Model with TensorFlow

In [ ]:
tf.random.set_seed(42)
nn = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_s.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(16, activation='relu'),
    Dense(1)
])

nn.compile(optimizer='adam', loss='mse', metrics=['mae'])
nn.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
history = nn.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=300, batch_size=16,
    callbacks=[early_stop], verbose=0
)
print(f'Training complete! Stopped at epoch {len(history.history["loss"])}')

nn_pred = nn.predict(X_test_s).flatten()
nn_r2   = r2_score(y_test, nn_pred)
nn_mae  = mean_absolute_error(y_test, nn_pred)
print(f'Neural Network → R²: {nn_r2:.4f} | MAE: {nn_mae:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['loss'],     label='Train', color='#3498db')
ax1.plot(history.history['val_loss'], label='Val',   color='#e74c3c')
ax1.set_title('Loss Curve'); ax1.legend()

ax2.plot(history.history['mae'],     label='Train', color='#3498db')
ax2.plot(history.history['val_mae'], label='Val',   color='#e74c3c')
ax2.set_title('MAE Curve'); ax2.legend()
plt.suptitle('Neural Network Training History', fontsize=13)
plt.tight_layout()
plt.show()

## Step 9: Budget Optimization Simulator

In [ ]:
print('=== 💡 Advertising Budget Simulator ===')
print('Predicting expected sales for different budget allocations:\n')

scenarios = [
    {'Name': 'TV-Heavy',     'TV': 200, 'Radio': 20,  'Newspaper': 10},
    {'Name': 'Radio-Heavy',  'TV': 50,  'Radio': 45,  'Newspaper': 10},
    {'Name': 'Balanced',     'TV': 100, 'Radio': 30,  'Newspaper': 20},
    {'Name': 'Newspaper-Focused', 'TV': 30, 'Radio': 10, 'Newspaper': 100},
    {'Name': 'Max Budget',   'TV': 250, 'Radio': 49,  'Newspaper': 50},
]

for sc in scenarios:
    row = pd.DataFrame([{
        'TV': sc['TV'], 'Radio': sc['Radio'], 'Newspaper': sc['Newspaper'],
        'TV_Radio': sc['TV']*sc['Radio'],
        'TV_sq': sc['TV']**2, 'Radio_sq': sc['Radio']**2,
        'Total_Spend': sc['TV']+sc['Radio']+sc['Newspaper'],
        'TV_share': sc['TV']/(sc['TV']+sc['Radio']+sc['Newspaper']+0.01)
    }])
    row_s = scaler.transform(row)
    pred  = best.predict(row_s)[0]
    total = sc['TV'] + sc['Radio'] + sc['Newspaper']
    print(f"  {sc['Name']:<20} | Budget: ${total}K | Predicted Sales: {pred:.2f} units")

print('\n✅ Task 4 Complete!')
print('\n=== Summary of Insights ===')
print('• TV advertising has the highest individual impact on sales')
print('• Radio spend has strong positive correlation with sales')
print('• Newspaper spending has relatively lower ROI')
print('• Combined TV + Radio investment yields best returns')
print('• Random Forest and Gradient Boosting outperform linear models')